In [1]:
import sqlite3
import math

In [ ]:
import sqlite3
import math

SRC_DB = "/home/joe/work/Fire/ML/Data/DB/wx_daily_2982.sqlite"
DST_DB = "/home/joe/work/Fire/ML/Data/DB/wx_daily_mean_2982.sqlite"

WIND_THRESHOLD = 7.0
TEMPS_ARE_KELVIN = True

def atan2_(y, x): return math.atan2(y, x)
def sqrt_(x): return math.sqrt(x)
def deg_(x): return x * 180.0 / math.pi
def vpd_kpa(T, Td):
    # if TEMPS_ARE_KELVIN:
    #     T -= 273.15
    #     Td -= 273.15
    es = 0.6108 * math.exp((17.27*T)/(T+237.3))
    ea = 0.6108 * math.exp((17.27*Td)/(Td+237.3))
    return es - ea
def rh_kpa(T, Td):
    # if TEMPS_ARE_KELVIN:
    #     T -= 273.15
    #     Td -= 273.15
    es = 0.6108 * math.exp((17.27*T)/(T+237.3))
    ea = 0.6108 * math.exp((17.27*Td)/(Td+237.3))
    rh = 100.0 * ea / es if es > 0 else 0.0
    return rh

conn = sqlite3.connect(DST_DB)
conn.create_function("SQRT", 1, sqrt_)
conn.create_function("ATAN2", 2, atan2_)
conn.create_function("DEG", 1, deg_)
conn.create_function("VPD_KPA", 2, vpd_kpa)
conn.create_function("RH_KPA", 2, rh_kpa)

cur = conn.cursor()
cur.execute(f"ATTACH DATABASE '{SRC_DB}' AS src")

cur.execute("""
CREATE TABLE IF NOT EXISTS daily_data (
    date TEXT NOT NULL,
    point_id INTEGER NOT NULL,
    T_mean REAL,
    Td_mean REAL,
    RH_mean REAL,
    VPD_mean REAL,
    VPD_max REAL,
    ws_mean REAL,
    ws_max REAL,
    ws_gust_count INTEGER,
    wd_mean REAL,
    pcp_sum REAL,
    PRIMARY KEY (date, point_id)
);
""")

conn.commit()

for year in range(1990, 2025):

    print(f"Processing year {year}")

    cur.execute(f"""
    WITH base AS (
        SELECT
            substr(datetime,1,10) AS date,
            point_id,
            variable,
            value
        FROM src.daily_data
        WHERE date >= '{year}-01-01'
          AND date <  '{year+1}-01-01'
    ),
    pivot AS (
        SELECT
            date,
            point_id,
            MAX(CASE WHEN variable='T'  THEN value-273.15 END) AS T,
            MAX(CASE WHEN variable='Td' THEN value-273.15 END) AS Td,
            MAX(CASE WHEN variable='RH' THEN value END) AS RH,
            MAX(CASE WHEN variable='pcp' THEN value END) AS pcp,
            MAX(CASE WHEN variable='u' THEN value END) AS u,
            MAX(CASE WHEN variable='v' THEN value END) AS v
        FROM base
        GROUP BY date, point_id, substr(date,1,13)
    ),
    calc AS (
        SELECT
            date,
            point_id,
            T, Td, RH_KPA(T,Td) AS RH,
            pcp,
            SQRT(u*u + v*v) AS ws,
            u, v,
            VPD_KPA(T,Td) AS vpd
        FROM pivot
    )
    INSERT OR REPLACE INTO daily_data
    SELECT
        date,
        point_id,
        AVG(T),
        AVG(Td),
        AVG(RH),
        AVG(vpd),
        MAX(vpd),
        AVG(ws),
        MAX(ws),
        SUM(CASE WHEN ws > {WIND_THRESHOLD} THEN 1 ELSE 0 END),
        (270 - DEG(ATAN2(AVG(v), AVG(u))) + 360) % 360,
        SUM(pcp)
    FROM calc
    GROUP BY date, point_id
    """)

    conn.commit()

conn.close()

print("Complete.")

Processing year 1990
Processing year 1991
Processing year 1992
Processing year 1993
Processing year 1994
Processing year 1995
Processing year 1996
Processing year 1997
Processing year 1998
Processing year 1999
Processing year 2000
Processing year 2001
Processing year 2002
Processing year 2003
Processing year 2004
Processing year 2005
Processing year 2006
Processing year 2007
Processing year 2008
Processing year 2009
Processing year 2010
Processing year 2011
Processing year 2012
Processing year 2013
Processing year 2014
Processing year 2015
Processing year 2016
Processing year 2017
Processing year 2018
Processing year 2019
Processing year 2020
Processing year 2021
Processing year 2022
Processing year 2023
Processing year 2024
Complete.


In [4]:
import sqlite3
import pandas as pd
DST_DB = "/home/joe/work/Fire/ML/Data/DB/wx_daily_mean_2982.sqlite"
conn = sqlite3.connect(DST_DB)

df = pd.read_sql_query("SELECT * FROM daily_data", conn)
print(df.head())

         date  point_id     T_mean    Td_mean    RH_mean  VPD_mean   VPD_max  \
0  1990-01-01         0 -10.763068 -13.532324  79.941039  0.053934  0.053934   
1  1990-01-01         1 -10.220099 -12.741309  81.658537  0.051496  0.051496   
2  1990-01-01         2 -10.974005 -12.799902  86.306058  0.036204  0.036204   
3  1990-01-01         3 -11.509161 -12.942480  89.051612  0.027729  0.027729   
4  1990-01-01         4 -11.829474 -13.085059  90.322415  0.023887  0.023887   

     ws_max  ws_gust_count  wd_mean  pcp_sum  
0  1.840061              0    188.0      0.0  
1  1.890359              0    200.0      0.0  
2  2.022510              0    211.0      0.0  
3  2.129501              0    216.0      0.0  
4  2.168313              0    215.0      0.0  


In [5]:
df.shape

(38121888, 11)

In [5]:
df['date'].value_counts()

date
2025-04-04    5964
1989-12-31    5964
1990-01-01    5964
1990-01-02    5964
1990-01-03    5964
              ... 
1990-01-18    5964
1990-01-19    5964
1990-01-20    5964
1990-01-21    5964
1990-01-22    5964
Name: count, Length: 12879, dtype: int64

In [7]:
df.columns

Index(['date', 'point_id', 'T_mean', 'Td_mean', 'RH_mean', 'VPD_mean',
       'VPD_max', 'ws_max', 'ws_gust_count', 'wd_mean', 'pcp_sum'],
      dtype='object')

In [10]:
df['point_id'].value_counts()

point_id
2981    12784
0       12784
1       12784
2       12784
3       12784
        ...  
9       12784
10      12784
11      12784
12      12784
13      12784
Name: count, Length: 2982, dtype: int64

In [11]:
365*(2024-1990+1)

12775

In [25]:
p1=df[df['point_id'] == 500]
p1['VPD_diff'] = p1['VPD_max'] - p1['VPD_mean']
p1.head()       

/tmp/ipykernel_2046/3661149723.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  p1['VPD_diff'] = p1['VPD_max'] - p1['VPD_mean']


,date,point_id,T_mean,Td_mean,RH_mean,VPD_mean,VPD_max,ws_max,ws_gust_count,wd_mean,pcp_sum,VPD_diff
500,1990-01-01,500,-8.204474,-11.225684,78.736929,0.069972,0.069972,0.634035,0,220.0,0.000000,0.0
3482,1990-01-02,500,-5.342581,-12.455588,57.182390,0.175700,0.175700,1.772836,0,202.0,0.000000,0.0
6464,1990-01-03,500,-3.599707,-5.202704,88.593319,0.053399,0.053399,2.378629,0,346.0,0.002293,0.0
9446,1990-01-04,500,-14.552542,-17.155585,80.449067,0.038642,0.038642,0.653358,0,151.0,0.000000,0.0
12428,1990-01-05,500,-14.391577,-18.577048,70.341071,0.059404,0.059404,0.787035,0,233.0,0.000002,0.0


In [27]:
p1['VPD_diff'].describe()

count    12784.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: VPD_diff, dtype: float64

In [ ]:
p1[p1['date'] < '1993-01-01'].plot(x='date', y='VPD_mean')

## NEW

In [2]:
SRC_DB = "/home/joe/work/Fire/ML/Data/DB/wx_daily_2982.sqlite"

conn = sqlite3.connect(DST_DB)

NameError: name 'DST_DB' is not defined

In [36]:
df = pd.read_sql_query("SELECT * FROM daily_data WHERE point_id = 500", conn)

In [37]:
df.head()

,date,point_id,T_mean,Td_mean,RH_mean,VPD_mean,VPD_max,ws_max,ws_gust_count,wd_mean,pcp_sum
0,1990-01-01,500,-8.204474,-11.225684,78.736929,0.069972,0.069972,0.634035,0,220.0,0.000000
1,1990-01-02,500,-5.342581,-12.455588,57.182390,0.175700,0.175700,1.772836,0,202.0,0.000000
2,1990-01-03,500,-3.599707,-5.202704,88.593319,0.053399,0.053399,2.378629,0,346.0,0.002293
3,1990-01-04,500,-14.552542,-17.155585,80.449067,0.038642,0.038642,0.653358,0,151.0,0.000000
4,1990-01-05,500,-14.391577,-18.577048,70.341071,0.059404,0.059404,0.787035,0,233.0,0.000002


In [5]:
import pandas as pd
SRC_DB = "/home/joe/work/Fire/ML/Data/DB/wx_daily_2982.sqlite"
DST_DB = "/home/joe/work/Fire/ML/Data/DB/wx_daily_mean_2982.sqlite"
SRC_DB = "/home/joe/work/Fire/ML/Data/DB/wx_daily_2982.sqlite"

conn = sqlite3.connect(DST_DB)


In [6]:
df = pd.read_sql_query("SELECT * FROM daily_data WHERE point_id = 500", conn)


DatabaseError: Execution failed on sql 'SELECT * FROM daily_data WHERE point_id = 500': no such table: daily_data